# 06. Feature Engineering V4 — 문헌 기반 성분 조합

**목적**  
문헌과 도메인 지식에 기반한 피부타입별 성분 조합 특징을 생성합니다.

**입력**  
`전성분_표준화_최종.csv와 V3_성분기능군_개수.csv`

**출력**  
`data/interim/V4_문헌기반_성분조합.csv`

> 저장된 전처리 데이터만 사용하며 외부 요청은 발생하지 않습니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


## V4 문헌 기반 특징 구성

피부유형별 성분의 지지·주의·상충 관계와 사용감 처방군을 수치화하고, V3 기능군 특징과 결합합니다.


## 1. 경로 설정


In [ ]:
from pathlib import Path
import hashlib
import math
import unicodedata

import numpy as np
import pandas as pd

LONG_PATH = DATA_INTERIM_DIR / "전성분_표준화_최종.csv"
V3_PATH = DATA_INTERIM_DIR / "V3_성분기능군_개수.csv"
OUTPUT_DIR = DATA_INTERIM_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_OUTPUT_PATH = OUTPUT_DIR / "V4_문헌기반_성분조합.csv"
DICTIONARY_OUTPUT_PATH = OUTPUT_DIR / "V4_특징설명.csv"

SKIN_TYPES = [
    "건성", "약건성", "지성", "복합성", "민감성", "트러블성", "중성"
]

print("전성분:", LONG_PATH)
print("V3:", V3_PATH)
print("출력:", OUTPUT_DIR)


## 2. 입력 데이터 검증 및 정리


In [ ]:
long_df = pd.read_csv(
    LONG_PATH,
    encoding='utf-8-sig',
)
v3_df = pd.read_csv(
    V3_PATH,
    encoding='utf-8-sig',
)

required_long = {
    'product_id',
    'product_name',
    'canonical_name',
    'ingredient_order',
}
required_v3 = {
    'product_id',
    'product_name',
}

missing_long = required_long - set(long_df.columns)
missing_v3 = required_v3 - set(v3_df.columns)

if missing_long:
    raise ValueError(
        f"전성분 필수 컬럼 누락: {sorted(missing_long)}"
    )

if missing_v3:
    raise ValueError(
        f"V3 필수 컬럼 누락: {sorted(missing_v3)}"
    )

# 두 파일에 product_group_id가 있으면 최종 통합 ID로 우선 사용
if (
    'product_group_id' in long_df.columns
    and 'product_group_id' in v3_df.columns
):
    ID_COL = 'product_group_id'
else:
    ID_COL = 'product_id'

long_df[ID_COL] = long_df[ID_COL].astype(str)
v3_df[ID_COL] = v3_df[ID_COL].astype(str)

long_df['canonical_name'] = (
    long_df['canonical_name']
    .astype('string')
    .str.strip()
)

long_df['ingredient_order'] = pd.to_numeric(
    long_df['ingredient_order'],
    errors='coerce',
)


In [2]:
long_df = long_df[
    long_df['canonical_name'].notna()
    & long_df['canonical_name'].ne('')
].copy()

ingredient_base = (
    long_df[
        [
            ID_COL,
            'product_name',
            'canonical_name',
            'ingredient_order',
        ]
    ]
    .sort_values(
        [ID_COL, 'canonical_name', 'ingredient_order']
    )
    .drop_duplicates(
        [ID_COL, 'canonical_name'],
        keep='first',
    )
    .reset_index(drop=True)
)

v3_count_columns = [
    column
    for column in v3_df.columns
    if column.startswith('v3_count_')
]

if not v3_count_columns:
    raise ValueError(
        "V3 파일에서 v3_count_ 컬럼을 찾지 못했습니다."
    )

long_ids = set(ingredient_base[ID_COL])
v3_ids = set(v3_df[ID_COL])

if long_ids != v3_ids:
    raise ValueError(
        "전성분과 V3의 제품 집합이 일치하지 않습니다.\n"
        f"전성분에만 존재: {sorted(long_ids - v3_ids)[:10]}\n"
        f"V3에만 존재: {sorted(v3_ids - long_ids)[:10]}"
    )

if v3_df[ID_COL].duplicated().any():
    raise ValueError("V3 파일에 제품 ID 중복이 있습니다.")

print("사용 ID:", ID_COL)
print("제품 수:", len(long_ids))
print("고유 성분 수:", ingredient_base['canonical_name'].nunique())
print("V3 Count 수:", len(v3_count_columns))


사용 ID: product_id
제품 수: 231
고유 성분 수: 1292
V3 Count 수: 12


## 3. 동일 전성분 처방 그룹


In [3]:

product_ingredient_sets = (
    ingredient_base
    .groupby(ID_COL)['canonical_name']
    .apply(set)
    .to_dict()
)

def formula_hash(ingredient_set):
    formula_text = '\x1f'.join(
        sorted(ingredient_set)
    )

    return hashlib.md5(
        formula_text.encode('utf-8')
    ).hexdigest()

formula_group_df = pd.DataFrame({
    ID_COL: list(product_ingredient_sets.keys()),
    'formula_group': [
        formula_hash(ingredient_set)
        for ingredient_set
        in product_ingredient_sets.values()
    ],
})

print(
    "동일 전성분 처방 그룹 수:",
    formula_group_df['formula_group'].nunique(),
)


동일 전성분 처방 그룹 수: 193


## 4. 주요 성분 포함·순서 Proxy


In [ ]:
product_max_order = (
    ingredient_base
    .groupby(ID_COL)['ingredient_order']
    .max()
    .to_dict()
)

def ingredient_order_feature(
    product_id,
    matcher,
):
    product_rows = ingredient_base[
        ingredient_base[ID_COL].eq(product_id)
    ]

    matched_rows = product_rows[
        product_rows['canonical_name'].map(matcher)
    ]

    if matched_rows.empty:
        return {
            'has': 0,
            'top10': 0,
            'prominence': 0.0,
        }

    min_order = matched_rows['ingredient_order'].min()
    max_order = product_max_order.get(
        product_id,
        np.nan,
    )

    if pd.isna(min_order):
        return {
            'has': 1,
            'top10': 0,
            'prominence': 0.0,
        }

    min_order = float(min_order)

    if pd.isna(max_order) or float(max_order) <= 1:
        prominence = 1.0
    else:
        prominence = 1.0 - (
            (min_order - 1.0)
            / (float(max_order) - 1.0)
        )

    return {
        'has': 1,
        'top10': int(min_order <= 10),
        'prominence': float(
            np.clip(prominence, 0.0, 1.0)
        ),
    }

order_rows = []


In [4]:
for product_id in sorted(product_ingredient_sets):
    niacinamide = ingredient_order_feature(
        product_id,
        lambda name: name == '나이아신아마이드',
    )
    panthenol = ingredient_order_feature(
        product_id,
        lambda name: name == '판테놀',
    )
    ceramide = ingredient_order_feature(
        product_id,
        lambda name: str(name).startswith('세라마이드'),
    )
    salicylic = ingredient_order_feature(
        product_id,
        lambda name: name in {
            '살리실릭애씨드',
            '카프릴로일살리실릭애씨드',
        },
    )

    order_rows.append({
        ID_COL: product_id,

        'v4_has_niacinamide':
            niacinamide['has'],
        'v4_niacinamide_top10':
            niacinamide['top10'],
        'v4_niacinamide_prominence':
            niacinamide['prominence'],

        'v4_has_panthenol':
            panthenol['has'],
        'v4_panthenol_top10':
            panthenol['top10'],
        'v4_panthenol_prominence':
            panthenol['prominence'],

        'v4_has_ceramide':
            ceramide['has'],
        'v4_ceramide_prominence':
            ceramide['prominence'],

        'v4_has_salicylic':
            salicylic['has'],
        'v4_salicylic_prominence':
            salicylic['prominence'],
    })

order_df = pd.DataFrame(order_rows)
display(order_df.head())


,product_id,v4_has_niacinamide,v4_niacinamide_top10,v4_niacinamide_prominence,v4_has_panthenol,v4_panthenol_top10,v4_panthenol_prominence,v4_has_ceramide,v4_ceramide_prominence,v4_has_salicylic,v4_salicylic_prominence
0,A000000002848,0,0,0.0,0,0,0.000000,0,0.0,1,0.43750
1,A000000010471,0,0,0.0,0,0,0.000000,0,0.0,1,0.26087
2,A000000117624,0,0,0.0,0,0,0.000000,0,0.0,0,0.00000
3,A000000126470,0,0,0.0,1,1,0.866667,0,0.0,0,0.00000
4,A000000134691,0,0,0.0,0,0,0.000000,0,0.0,0,0.00000


## 5. 사용감 처방군 Count


In [ ]:
ESSENTIAL_OIL_KEYWORDS = [
    '라벤더',
    '베르가모트',
    '티트리',
    '로즈마리',
    '페퍼민트',
    '유칼립투스',
    '레몬',
    '라임',
    '오렌지',
    '자몽',
    '만다린',
    '일랑일랑',
    '클라리',
    '제라늄',
]

SENSORY_GROUPS = [
    'light_ester',
    'volatile_silicone',
    'nonvolatile_silicone',
    'hydrocarbon_occlusive',
    'fatty_alcohol',
    'plant_oil',
    'wax_butter',
    'essential_oil',
]


In [ ]:
def classify_sensory_group(name):
    name = str(name)
    groups = set()

    if any(token in name for token in [
        '세틸에틸헥사노에이트',
        '다이카프릴릴카보네이트',
        '다이카프릴릴에터',
        '트라이에틸헥사노인',
        '아이소노닐아이소노나노에이트',
        '아이소프로필미리스테이트',
        'C12-15알킬벤조에이트',
        '코코-카프릴레이트/카프레이트',
    ]):
        groups.add('light_ester')

    if any(token in name for token in [
        '사이클로펜타실록세인',
        '사이클로헥사실록세인',
        '메틸트라이메티콘',
        '트라이실록세인',
        '카프릴릴메티콘',
    ]):
        groups.add('volatile_silicone')

    if any(token in name for token in [
        '다이메티콘',
        '다이메티콘올',
        '비닐다이메티콘',
        '폴리메틸실세스퀴옥세인',
    ]):
        groups.add('nonvolatile_silicone')

    if any(token in name for token in [
        '페트롤라툼',
        '미네랄오일',
        '파라핀',
        '하이드로제네이티드폴리데센',
        '하이드로제네이티드폴리아이소부텐',
    ]):
        groups.add('hydrocarbon_occlusive')

    if any(token in name for token in [
        '세테아릴알코올',
        '세틸알코올',
        '스테아릴알코올',
        '베헤닐알코올',
        '아라키딜알코올',
    ]):
        groups.add('fatty_alcohol')

    is_essential_oil = (
        name.endswith('오일')
        and any(
            keyword in name
            for keyword in ESSENTIAL_OIL_KEYWORDS
        )
    )

    if is_essential_oil:
        groups.add('essential_oil')
    elif name.endswith('오일'):
        groups.add('plant_oil')

    if (
        name.endswith('버터')
        or '왁스' in name
        or name in {'세레신', '오조케라이트'}
    ):
        groups.add('wax_butter')

    return groups


In [5]:
sensory_long = (
    ingredient_base[
        [ID_COL, 'canonical_name']
    ]
    .drop_duplicates()
    .copy()
)

sensory_long['sensory_group'] = (
    sensory_long['canonical_name']
    .map(classify_sensory_group)
)

sensory_long = sensory_long[
    sensory_long['sensory_group'].map(bool)
].explode('sensory_group')

sensory_wide = (
    sensory_long
    .groupby(
        [ID_COL, 'sensory_group']
    )['canonical_name']
    .nunique()
    .unstack(fill_value=0)
    .reindex(
        columns=SENSORY_GROUPS,
        fill_value=0,
    )
)

sensory_wide.columns = [
    f'v4_sensory_count_{group}'
    for group in sensory_wide.columns
]

sensory_wide = sensory_wide.reset_index()
display(sensory_wide.head())


,product_id,v4_sensory_count_light_ester,v4_sensory_count_volatile_silicone,v4_sensory_count_nonvolatile_silicone,v4_sensory_count_hydrocarbon_occlusive,v4_sensory_count_fatty_alcohol,v4_sensory_count_plant_oil,v4_sensory_count_wax_butter,v4_sensory_count_essential_oil
0,A000000002848,0,2,3,1,0,1,0,0
1,A000000010471,0,0,2,0,0,0,0,0
2,A000000117624,0,0,0,1,0,1,2,0
3,A000000126470,0,0,0,0,0,1,1,0
4,A000000134691,0,0,0,1,0,1,0,0


## 6. 문헌 규칙 계산용 제품 Facts


In [ ]:
CHOLESTEROL_NAMES = {
    '콜레스테롤',
}

FATTY_ACID_NAMES = {
    '스테아릭애씨드',
    '팔미틱애씨드',
    '리놀레익애씨드',
    '리놀레닉애씨드',
}

FRAGRANCE_ALLERGEN_NAMES = {
    '향료',
    '리모넨',
    '리날룰',
    '시트랄',
    '시트로넬올',
    '제라니올',
    '유제놀',
    '아이소유제놀',
    '쿠마린',
    '파네솔',
    '벤질살리실레이트',
    '벤질벤조에이트',
    '헥실신남알',
    '알파-아이소메틸아이오논',
    '아니스알코올',
}

ACID_EXFOLIANT_NAMES = {
    '살리실릭애씨드',
    '카프릴로일살리실릭애씨드',
    '글라이콜릭애씨드',
    '락틱애씨드',
    '만델릭애씨드',
    '글루코노락톤',
    '락토바이오닉애씨드',
    '아젤라익애씨드',
    '석시닉애씨드',
}

fact_rows = []


In [6]:
for product_id, ingredients in (
    product_ingredient_sets.items()
):
    fact_rows.append({
        ID_COL: product_id,

        'v4_has_cholesterol': int(
            bool(ingredients & CHOLESTEROL_NAMES)
        ),
        'v4_has_fatty_acid': int(
            bool(ingredients & FATTY_ACID_NAMES)
        ),

        # 민감성 향료 규칙 계산에만 사용.
        # v3_count_fragrance와 중복되므로 최종 X에는 저장하지 않음.
        '_internal_fragrance_allergen_count': len(
            ingredients & FRAGRANCE_ALLERGEN_NAMES
        ),

        'v4_acid_exact_count': len(
            ingredients & ACID_EXFOLIANT_NAMES
        ),
    })

fact_df = pd.DataFrame(fact_rows)
display(fact_df.head())


,product_id,v4_has_cholesterol,v4_has_fatty_acid,_internal_fragrance_allergen_count,v4_acid_exact_count
0,A000000002848,0,0,1,1
1,A000000010471,0,0,1,2
2,A000000117624,0,1,1,0
3,A000000126470,0,0,0,0
4,A000000134691,0,0,0,0


## 7. 제품×피부유형 기본 데이터 생성


In [ ]:
product_base = (
    v3_df[
        [ID_COL, 'product_name']
        + v3_count_columns
    ]
    .merge(
        formula_group_df,
        on=ID_COL,
        how='inner',
        validate='one_to_one',
    )
    .merge(
        order_df,
        on=ID_COL,
        how='left',
        validate='one_to_one',
    )
    .merge(
        sensory_wide,
        on=ID_COL,
        how='left',
        validate='one_to_one',
    )
    .merge(
        fact_df,
        on=ID_COL,
        how='left',
        validate='one_to_one',
    )
)

product_numeric_columns = [
    column
    for column in product_base.columns
    if column not in {
        ID_COL,
        'product_name',
        'formula_group',
    }
]

product_base[product_numeric_columns] = (
    product_base[product_numeric_columns]
    .fillna(0)
)

skin_df = pd.DataFrame({
    '피부타입': SKIN_TYPES,
})

product_base['_join_key'] = 1
skin_df['_join_key'] = 1


In [7]:
x_df = (
    product_base
    .merge(
        skin_df,
        on='_join_key',
        how='inner',
    )
    .drop(columns='_join_key')
)

skin_dummies = pd.get_dummies(
    x_df['피부타입'],
    prefix='skin',
    dtype='int8',
)

x_df = pd.concat(
    [
        x_df.reset_index(drop=True),
        skin_dummies.reset_index(drop=True),
    ],
    axis=1,
)

print("제품×피부유형 행:", len(x_df))
print("제품 수:", x_df[ID_COL].nunique())
print("피부유형 수:", x_df['피부타입'].nunique())


제품×피부유형 행: 1617
제품 수: 231
피부유형 수: 7


## 8. 피부유형 조건부 문헌 규칙


In [ ]:
RULE_DEFINITIONS = [
    {
        'rule_id': 'dry_hydration_completeness',
        'block': 'clinical',
        'skin_types': ['건성', '약건성'],
        'direction': 'support',
        'formula_key': 'dry_hydration_completeness',
        'evidence_weight': 1.00,
        'description': (
            '건성·약건성의 보습·유연·밀폐 구성 완성도'
        ),
        'source_id': 'PMID:31532576',
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/31532576/'
        ),
    },
    {
        'rule_id': 'dry_barrier_lipid_completeness',
        'block': 'clinical',
        'skin_types': ['건성', '약건성'],
        'direction': 'support',
        'formula_key': 'dry_barrier_lipid_completeness',
        'evidence_weight': 0.64,
        'description': (
            '세라마이드·콜레스테롤·지방산 '
            '장벽지질 구성 완성도'
        ),
        'source_id': 'PMID:8618046',
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/8618046/'
        ),
    },
    {
        'rule_id': 'sensitive_recovery_system',
        'block': 'clinical',
        'skin_types': ['민감성'],
        'direction': 'support',
        'formula_key': 'sensitive_recovery_system',
        'evidence_weight': 0.48,
        'description': (
            '판테놀 또는 나이아신아마이드와 '
            '장벽·진정 기능군의 조합'
        ),
        'source_id': 'PMID:21982351',
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/21982351/'
        ),
    },
    {
        'rule_id': 'sensitive_fragrance_load',
        'block': 'caution',
        'skin_types': ['민감성'],
        'direction': 'caution',
        'formula_key': 'sensitive_fragrance_load',
        'evidence_weight': 0.80,
        'description': '민감성 피부의 향료 부담',
        'source_id': (
            'SCCS fragrance allergens; PMID:14572300'
        ),
        'source_url': (
            'https://health.ec.europa.eu/'
            'scientific-committees/'
            'scientific-committee-consumer-safety-sccs_en'
        ),
    },
    {
        'rule_id': 'sensitive_acid_load',
        'block': 'caution',
        'skin_types': ['민감성'],
        'direction': 'caution',
        'formula_key': 'sensitive_acid_load',
        'evidence_weight': 0.448,
        'description': '민감성 피부의 산·각질 성분 부담',
        'source_id': 'PMID:15725565',
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/15725565/'
        ),
    },
    {
        'rule_id': 'sensitive_essential_oil_load',
        'block': 'caution',
        'skin_types': ['민감성'],
        'direction': 'caution',
        'formula_key': 'sensitive_essential_oil_load',
        'evidence_weight': 0.288,
        'description': '민감성 피부의 에센셜오일 부담',
        'source_id': 'PMID:24175401',
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/24175401/'
        ),
    },
    {
        'rule_id': 'oily_niacinamide_prominence',
        'block': 'clinical',
        'skin_types': ['지성'],
        'direction': 'support',
        'formula_key': 'oily_niacinamide_prominence',
        'evidence_weight': 0.80,
        'description': (
            '지성 피부에서 나이아신아마이드 '
            '표기 prominence'
        ),
        'source_id': 'PMID:16766489',
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/16766489/'
        ),
    },
    {
        'rule_id': 'oily_heavy_load',
        'block': 'sensory',
        'skin_types': ['지성'],
        'direction': 'exploratory',
        'formula_key': 'oily_heavy_load',
        'evidence_weight': 0.00,
        'description': (
            '왁스·버터·식물성 오일·탄화수소 '
            '밀폐제·지방알코올 기반 Heavy 처방 Proxy'
        ),
        'source_id': (
            'PMID:16116522; PMID:22085371'
        ),
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/16116522/'
        ),
    },
    {
        'rule_id': 'oily_light_system',
        'block': 'sensory',
        'skin_types': ['지성'],
        'direction': 'exploratory',
        'formula_key': 'oily_light_system',
        'evidence_weight': 0.00,
        'description': (
            '휘발성 실리콘·가벼운 에스터·'
            '실리콘 기반 Light 처방 Proxy'
        ),
        'source_id': (
            'PMID:16116522; PMID:22085371'
        ),
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/22085371/'
        ),
    },
    {
        'rule_id': 'trouble_niacinamide_ceramide',
        'block': 'clinical',
        'skin_types': ['트러블성'],
        'direction': 'support',
        'formula_key': 'trouble_niacinamide_ceramide',
        'evidence_weight': 0.70,
        'description': (
            '트러블성 피부의 '
            '나이아신아마이드·세라마이드 동시 포함'
        ),
        'source_id': 'PMID:38299457',
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/38299457/'
        ),
    },
    {
        'rule_id': 'trouble_salicylic_barrier',
        'block': 'clinical',
        'skin_types': ['트러블성'],
        'direction': 'support',
        'formula_key': 'trouble_salicylic_barrier',
        'evidence_weight': 0.20,
        'description': (
            '살리실릭애씨드와 장벽 기능군의 '
            '보상형 조합 가설'
        ),
        'source_id': (
            'PMID:15725565; PMID:38299457'
        ),
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/38299457/'
        ),
    },
    {
        'rule_id': 'acid_without_barrier_compensation',
        'block': 'caution',
        'skin_types': ['민감성', '트러블성'],
        'direction': 'caution',
        'formula_key': 'acid_without_barrier_compensation',
        'evidence_weight': 0.20,
        'description': (
            '산 성분은 있으나 장벽 기능군 구성이 낮은 경우'
        ),
        'source_id': 'PMID:15725565',
        'source_url': (
            'https://pubmed.ncbi.nlm.nih.gov/15725565/'
        ),
    },
]


In [8]:
rule_df = pd.DataFrame(RULE_DEFINITIONS)
display(
    rule_df[
        [
            'rule_id',
            'block',
            'skin_types',
            'direction',
            'evidence_weight',
            'source_id',
        ]
    ]
)


,rule_id,block,skin_types,direction,evidence_weight,source_id
0,dry_hydration_completeness,clinical,"[건성, 약건성]",support,1.000,PMID:31532576
1,dry_barrier_lipid_completeness,clinical,"[건성, 약건성]",support,0.640,PMID:8618046
2,sensitive_recovery_system,clinical,[민감성],support,0.480,PMID:21982351
3,sensitive_fragrance_load,caution,[민감성],caution,0.800,SCCS fragrance allergens; PMID:14572300
4,sensitive_acid_load,caution,[민감성],caution,0.448,PMID:15725565
5,sensitive_essential_oil_load,caution,[민감성],caution,0.288,PMID:24175401
6,oily_niacinamide_prominence,clinical,[지성],support,0.800,PMID:16766489
7,oily_heavy_load,sensory,[지성],exploratory,0.000,PMID:16116522; PMID:22085371
8,oily_light_system,sensory,[지성],exploratory,0.000,PMID:16116522; PMID:22085371
9,trouble_niacinamide_ceramide,clinical,[트러블성],support,0.700,PMID:38299457


In [ ]:
def saturated_count(value, scale):
    value = max(float(value), 0.0)

    return 1.0 - math.exp(
        -value / scale
    )

def geometric_mean(values):
    values = [
        max(float(value), 0.0)
        for value in values
    ]

    if any(value <= 0 for value in values):
        return 0.0

    return float(
        np.prod(values)
        ** (1.0 / len(values))
    )


In [ ]:
def raw_rule_strength(row, formula_key):
    if formula_key == 'dry_hydration_completeness':
        return geometric_mean([
            saturated_count(
                row['v3_count_humectant'], 3
            ),
            saturated_count(
                row['v3_count_emollient'], 2
            ),
            saturated_count(
                row['v3_count_occlusive'], 2
            ),
        ])

    if formula_key == 'dry_barrier_lipid_completeness':
        return (
            row['v4_has_ceramide']
            + row['v4_has_cholesterol']
            + row['v4_has_fatty_acid']
        ) / 3.0

    if formula_key == 'sensitive_recovery_system':
        key_active = max(
            row['v4_has_panthenol'],
            row['v4_has_niacinamide'],
        )

        return geometric_mean([
            key_active,
            saturated_count(
                row['v3_count_barrier'], 2
            ),
            saturated_count(
                row['v3_count_soothing'], 2
            ),
        ])

    if formula_key == 'sensitive_fragrance_load':
        return saturated_count(
            max(
                row['v3_count_fragrance'],
                row['_internal_fragrance_allergen_count'],
            ),
            1,
        )

    if formula_key == 'sensitive_acid_load':
        return saturated_count(
            max(
                row['v3_count_acid_exfoliant'],
                row['v4_acid_exact_count'],
            ),
            1,
        )

    if formula_key == 'sensitive_essential_oil_load':
        return saturated_count(
            row['v4_sensory_count_essential_oil'],
            1,
        )

    if formula_key == 'oily_niacinamide_prominence':
        return float(
            row['v4_niacinamide_prominence']
        )

    if formula_key == 'oily_heavy_load':
        return float(np.mean([
            saturated_count(
                row['v4_sensory_count_wax_butter'], 1
            ),
            saturated_count(
                row['v4_sensory_count_plant_oil'], 2
            ),
            saturated_count(
                row[
                    'v4_sensory_count_'
                    'hydrocarbon_occlusive'
                ],
                1,
            ),
            saturated_count(
                row['v4_sensory_count_fatty_alcohol'], 2
            ),
        ]))

    if formula_key == 'oily_light_system':
        return float(np.mean([
            saturated_count(
                row[
                    'v4_sensory_count_'
                    'volatile_silicone'
                ],
                1,
            ),
            saturated_count(
                row['v4_sensory_count_light_ester'], 2
            ),
            saturated_count(
                row[
                    'v4_sensory_count_'
                    'nonvolatile_silicone'
                ],
                2,
            ),
        ]))

    if formula_key == 'trouble_niacinamide_ceramide':
        return float(
            row['v4_has_niacinamide']
            and row['v4_has_ceramide']
        )

    if formula_key == 'trouble_salicylic_barrier':
        return geometric_mean([
            float(row['v4_has_salicylic']),
            saturated_count(
                row['v3_count_barrier'], 2
            ),
        ])

    if formula_key == 'acid_without_barrier_compensation':
        acid_strength = saturated_count(
            max(
                row['v3_count_acid_exfoliant'],
                row['v4_acid_exact_count'],
            ),
            1,
        )

        barrier_strength = saturated_count(
            row['v3_count_barrier'],
            2,
        )

        return acid_strength * (
            1.0 - barrier_strength
        )

    raise KeyError(
        f"정의되지 않은 규칙: {formula_key}"
    )


In [ ]:
support_terms = []
caution_terms = []

for rule in RULE_DEFINITIONS:
    feature_name = (
        f"v4_rule_{rule['rule_id']}"
    )

    skin_match = (
        x_df['피부타입']
        .isin(rule['skin_types'])
        .astype(float)
    )

    raw_strength = x_df.apply(
        lambda row: raw_rule_strength(
            row,
            rule['formula_key'],
        ),
        axis=1,
    )

    x_df[feature_name] = (
        skin_match * raw_strength
    )

    if (
        rule['direction'] == 'support'
        and rule['evidence_weight'] > 0
    ):
        support_terms.append((
            feature_name,
            float(rule['evidence_weight']),
        ))

    if (
        rule['direction'] == 'caution'
        and rule['evidence_weight'] > 0
    ):
        caution_terms.append((
            feature_name,
            float(rule['evidence_weight']),
        ))

# 개별 weighted 컬럼은 만들지 않고
# 근거 가중치는 집계점수 계산에만 사용
x_df['v4_evidence_support_score'] = sum(
    x_df[column] * weight
    for column, weight in support_terms
)

x_df['v4_evidence_caution_score'] = sum(
    x_df[column] * weight
    for column, weight in caution_terms
)

# net은 support-caution의 선형결합이라 생성하지 않음


In [9]:
x_df['v4_evidence_conflict_score'] = (
    x_df['v4_evidence_support_score']
    * x_df['v4_evidence_caution_score']
)


## 9. Clean X 컬럼 선정 및 자동 정리


In [ ]:
id_columns = [
    ID_COL,
    'product_name',
    '피부타입',
    'formula_group',
]

skin_columns = [
    column
    for column in x_df.columns
    if column.startswith('skin_')
]

sensory_columns = [
    f'v4_sensory_count_{group}'
    for group in SENSORY_GROUPS
]

presence_order_columns = [
    'v4_has_niacinamide',
    'v4_niacinamide_top10',
    'v4_niacinamide_prominence',

    'v4_has_panthenol',
    'v4_panthenol_top10',
    'v4_panthenol_prominence',

    'v4_has_ceramide',
    'v4_ceramide_prominence',

    'v4_has_salicylic',
    'v4_salicylic_prominence',

    'v4_has_cholesterol',
    'v4_has_fatty_acid',
    'v4_acid_exact_count',
]

rule_columns = [
    f"v4_rule_{rule['rule_id']}"
    for rule in RULE_DEFINITIONS
]

evidence_summary_columns = [
    'v4_evidence_support_score',
    'v4_evidence_caution_score',
    'v4_evidence_conflict_score',
]

# 우선순위가 높은 피처를 앞에 배치.
# 완전중복이 발생하면 먼저 나온 피처를 유지.
candidate_x_columns = (
    skin_columns
    + v3_count_columns
    + sensory_columns
    + presence_order_columns
    + rule_columns
    + evidence_summary_columns
)


In [ ]:
final_x = x_df[
    id_columns + candidate_x_columns
].copy()

# 1) 상수 피처 자동 제거
constant_features = [
    column
    for column in candidate_x_columns
    if final_x[column].nunique(
        dropna=False
    ) <= 1
]

if constant_features:
    final_x = final_x.drop(
        columns=constant_features
    )

remaining_x_columns = [
    column
    for column in candidate_x_columns
    if column not in constant_features
]

# 2) 값이 완전히 동일한 중복 피처 자동 제거
duplicate_feature_map = {}
kept_x_columns = []

for column in remaining_x_columns:
    duplicate_of = None

    for kept_column in kept_x_columns:
        if final_x[column].equals(
            final_x[kept_column]
        ):
            duplicate_of = kept_column
            break

    if duplicate_of is None:
        kept_x_columns.append(column)
    else:
        duplicate_feature_map[column] = duplicate_of

if duplicate_feature_map:
    final_x = final_x.drop(
        columns=list(duplicate_feature_map)
    )

final_x = final_x[
    id_columns + kept_x_columns
]

# 최종 검증
assert final_x[
    [ID_COL, '피부타입']
].duplicated().sum() == 0

assert final_x[
    kept_x_columns
].isna().sum().sum() == 0


In [10]:
expected_rows = (
    final_x[ID_COL].nunique()
    * len(SKIN_TYPES)
)

assert len(final_x) == expected_rows

print("제거된 상수 피처:", constant_features)
print("제거된 완전중복 피처:", duplicate_feature_map)
print("최종 행 수:", len(final_x))
print("ID 컬럼 수:", len(id_columns))
print("최종 실제 X 변수 수:", len(kept_x_columns))
print("최종 전체 컬럼 수:", len(final_x.columns))

display(final_x.head(10))


제거된 상수 피처: []
제거된 완전중복 피처: {}
최종 행 수: 1617
ID 컬럼 수: 4
최종 실제 X 변수 수: 55
최종 전체 컬럼 수: 59


,product_id,product_name,피부타입,formula_group,skin_건성,skin_민감성,skin_복합성,skin_약건성,skin_중성,skin_지성,...,v4_rule_sensitive_essential_oil_load,v4_rule_oily_niacinamide_prominence,v4_rule_oily_heavy_load,v4_rule_oily_light_system,v4_rule_trouble_niacinamide_ceramide,v4_rule_trouble_salicylic_barrier,v4_rule_acid_without_barrier_compensation,v4_evidence_support_score,v4_evidence_caution_score,v4_evidence_conflict_score
0,A000000002848,바이오더마 세비엄 포어 리파이너,건성,f5f6b9ab23517e277ec36cdb5c3aeb67,1,0,0,0,0,0,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.703318,0.000000,0.0
1,A000000002848,바이오더마 세비엄 포어 리파이너,약건성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,0,1,0,0,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.703318,0.000000,0.0
2,A000000002848,바이오더마 세비엄 포어 리파이너,지성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,0,0,0,1,...,0.0,0.0,0.256397,0.547178,0.0,0.0,0.000000,0.000000,0.000000,0.0
3,A000000002848,바이오더마 세비엄 포어 리파이너,복합성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,1,0,0,0,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0
4,A000000002848,바이오더마 세비엄 포어 리파이너,민감성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,1,0,0,0,0,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.632121,0.000000,0.915311,0.0
5,A000000002848,바이오더마 세비엄 포어 리파이너,트러블성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,0,0,0,0,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.632121,0.000000,0.126424,0.0
6,A000000002848,바이오더마 세비엄 포어 리파이너,중성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,0,0,1,0,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0
7,A000000010471,라로슈포제 에빠끌라 MAT 세보 컨트롤링 모이스춰라이저,건성,5d2afbc766b629825e95fa18cc3ab152,1,0,0,0,0,0,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.539722,0.000000,0.0
8,A000000010471,라로슈포제 에빠끌라 MAT 세보 컨트롤링 모이스춰라이저,약건성,5d2afbc766b629825e95fa18cc3ab152,0,0,0,1,0,0,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.539722,0.000000,0.0
9,A000000010471,라로슈포제 에빠끌라 MAT 세보 컨트롤링 모이스춰라이저,지성,5d2afbc766b629825e95fa18cc3ab152,0,0,0,0,0,1,...,0.0,0.0,0.000000,0.210707,0.0,0.0,0.000000,0.000000,0.000000,0.0


## 10. 최종 Feature Dictionary


In [ ]:
dictionary_rows = []

def add_dictionary_row(
    feature_name,
    feature_block,
    description,
    skin_conditioned,
    direction='model_learns',
    evidence_weight='',
    source_id='',
    source_url='',
):
    dictionary_rows.append({
        'feature_name': feature_name,
        'feature_block': feature_block,
        'description': description,
        'skin_conditioned': skin_conditioned,
        'direction': direction,
        'evidence_weight': evidence_weight,
        'source_id': source_id,
        'source_url': source_url,
    })

# 피부유형
for column in skin_columns:
    add_dictionary_row(
        column,
        'skin_type',
        '피부유형 One-hot',
        True,
        source_id='review skin type',
    )

# V3
for column in v3_count_columns:
    add_dictionary_row(
        column,
        'V3_function_count',
        '제품에 포함된 해당 기능군의 고유 성분 개수',
        False,
        source_id='V3 functional ingredient mapping',
    )

# 사용감
for group in SENSORY_GROUPS:
    add_dictionary_row(
        f'v4_sensory_count_{group}',
        'sensory_base',
        f'{group} 감각 처방군의 고유 성분 개수',
        False,
        source_id='PMID:16116522; PMID:22085371',
        source_url=(
            'https://pubmed.ncbi.nlm.nih.gov/16116522/'
        ),
    )

# 주요 성분·순서


In [ ]:
presence_order_descriptions = {
    'v4_has_niacinamide':
        '나이아신아마이드 포함 여부',
    'v4_niacinamide_top10':
        '나이아신아마이드 상위 10성분 표기 여부',
    'v4_niacinamide_prominence':
        '나이아신아마이드 표기 순서 Proxy',

    'v4_has_panthenol':
        '판테놀 포함 여부',
    'v4_panthenol_top10':
        '판테놀 상위 10성분 표기 여부',
    'v4_panthenol_prominence':
        '판테놀 표기 순서 Proxy',

    'v4_has_ceramide':
        '세라마이드 계열 포함 여부',
    'v4_ceramide_prominence':
        '세라마이드 계열 표기 순서 Proxy',

    'v4_has_salicylic':
        '살리실릭애씨드 계열 포함 여부',
    'v4_salicylic_prominence':
        '살리실릭애씨드 계열 표기 순서 Proxy',

    'v4_has_cholesterol':
        '콜레스테롤 포함 여부',
    'v4_has_fatty_acid':
        '선정 지방산 포함 여부',
    'v4_acid_exact_count':
        '선정 산·각질 성분의 고유 개수',
}

for column, description in (
    presence_order_descriptions.items()
):
    add_dictionary_row(
        column,
        'ingredient_presence_order',
        description,
        False,
        source_id='ingredient list',
    )

# 피부유형 조건부 규칙
for rule in RULE_DEFINITIONS:
    add_dictionary_row(
        f"v4_rule_{rule['rule_id']}",
        rule['block'],
        rule['description'],
        True,
        direction=rule['direction'],
        evidence_weight=rule['evidence_weight'],
        source_id=rule['source_id'],
        source_url=rule['source_url'],
    )

# 집계점수


In [11]:
add_dictionary_row(
    'v4_evidence_support_score',
    'evidence_summary',
    'Support 방향 규칙 강도×근거 신뢰도의 합',
    True,
    source_id='derived from evidence rules',
)
add_dictionary_row(
    'v4_evidence_caution_score',
    'evidence_summary',
    'Caution 방향 규칙 강도×근거 신뢰도의 합',
    True,
    source_id='derived from evidence rules',
)
add_dictionary_row(
    'v4_evidence_conflict_score',
    'evidence_summary',
    'Support와 Caution 점수가 동시에 큰 정도',
    True,
    source_id='derived from evidence rules',
)

feature_dictionary = pd.DataFrame(
    dictionary_rows
).drop_duplicates(
    subset='feature_name',
)

# 자동 정리 후 실제로 남은 X만 사전에 유지
feature_dictionary = feature_dictionary[
    feature_dictionary['feature_name']
    .isin(kept_x_columns)
].copy()

# X 데이터 순서와 동일하게 정렬
feature_order = {
    feature: index
    for index, feature
    in enumerate(kept_x_columns)
}

feature_dictionary['_order'] = (
    feature_dictionary['feature_name']
    .map(feature_order)
)

feature_dictionary = (
    feature_dictionary
    .sort_values('_order')
    .drop(columns='_order')
    .reset_index(drop=True)
)

assert len(feature_dictionary) == len(
    kept_x_columns
)

display(feature_dictionary.head(20))


,feature_name,feature_block,description,skin_conditioned,direction,evidence_weight,source_id,source_url
0,skin_건성,skin_type,피부유형 One-hot,True,model_learns,,review skin type,
1,skin_민감성,skin_type,피부유형 One-hot,True,model_learns,,review skin type,
2,skin_복합성,skin_type,피부유형 One-hot,True,model_learns,,review skin type,
3,skin_약건성,skin_type,피부유형 One-hot,True,model_learns,,review skin type,
4,skin_중성,skin_type,피부유형 One-hot,True,model_learns,,review skin type,
5,skin_지성,skin_type,피부유형 One-hot,True,model_learns,,review skin type,
6,skin_트러블성,skin_type,피부유형 One-hot,True,model_learns,,review skin type,
7,v3_count_humectant,V3_function_count,제품에 포함된 해당 기능군의 고유 성분 개수,False,model_learns,,V3 functional ingredient mapping,
8,v3_count_emollient,V3_function_count,제품에 포함된 해당 기능군의 고유 성분 개수,False,model_learns,,V3 functional ingredient mapping,
9,v3_count_occlusive,V3_function_count,제품에 포함된 해당 기능군의 고유 성분 개수,False,model_learns,,V3 functional ingredient mapping,


## 11. 저장 및 최종 QA


In [ ]:

final_x.to_csv(
    FEATURE_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)

feature_dictionary.to_csv(
    DICTIONARY_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)

print("저장 완료")
print("1.", FEATURE_OUTPUT_PATH)
print("2.", DICTIONARY_OUTPUT_PATH)

print("\n최종 QA")
print("- 제품 수:", final_x[ID_COL].nunique())
print("- 피부유형 수:", final_x['피부타입'].nunique())
print("- 총 행 수:", len(final_x))
print("- 처방 그룹 수:", final_x['formula_group'].nunique())
print("- 실제 X 변수 수:", len(kept_x_columns))
print("- Dictionary 행 수:", len(feature_dictionary))
print("- 결측값:", final_x.isna().sum().sum())
print(
    "- 제품×피부유형 중복:",
    final_x[
        [ID_COL, '피부타입']
    ].duplicated().sum(),
)

for forbidden_prefix in [
    'v4_weighted_',
]:
    assert not any(
        column.startswith(forbidden_prefix)
        for column in final_x.columns
    )

for forbidden_column in [
    'v4_fragrance_allergen_count',
    'v4_evidence_net_score',
    'negative_rate',
    'high_risk',
    'target',
]:
    assert forbidden_column not in final_x.columns

print("- 금지 컬럼 검증: 정상")


## 12. 모델링 데이터 결합 기준

리뷰 Target과 `product_id`, `피부타입`을 기준으로 결합하며 `v3_`, `v4_`, `skin_` 접두사 컬럼을 입력 특징으로 사용합니다.
